<a href="https://colab.research.google.com/github/DDarion/compressed-gaussians-colab/blob/main/ECNG_3020_3DGS_Viewing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Environment Set-up
## Confirming the GPU being used

In [ ]:
!nvidia-smi

Fri Nov 13 16:28:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Version comparions

In [ ]:
# nvcc and pytorch cuda version
%cd /content/gaussian-splatting
!nvcc --version
import torch
print(torch.__version__, torch.version.cuda)
!pip install -v --no-build-isolation submodules/diff-gaussian-rasterization 2>&1 | tail -60

[Errno 2] No such file or directory: '/content/gaussian-splatting'
/content
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Aug_20_01:58:59_PM_PDT_2025
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0
2.14.1+cu130 13.0
Using pip 26.2.2 from /usr/local/lib/python3.13/dist-packages/pip (python 3.13)
ERROR: Invalid requirement: 'submodules/diff-gaussian-rasterization': Expected semicolon (after name with no version specifier) or end
    submodules/diff-gaussian-rasterization
              ^
Hint: It looks like a path. File 'submodules/diff-gaussian-rasterization' does not exist.


# Setting up baseline Kerbl model
## Viewing


In [1]:
# this is as simple as drag and dropping.
# when this code runs, a url should open
# drag the .ply file from the training module and put in on the opened url
# the splat should show from there

# you get files to start the website. this is where the website is started from.
# installing the tunnel tool which is cloudflared. cloudflared attaches the website with an address so that it can be seen.

%cd /content
!git clone https://github.com/camenduru/splat
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -O /content/cloudflared-linux-amd64.deb
!dpkg -i /content/cloudflared-linux-amd64.deb
!which cloudflared

import atexit, requests, subprocess, time, re
from random import randint

# start the website here
server = subprocess.Popen(['python', '-m', 'http.server', '7860'], cwd='/content/splat')
atexit.register(server.terminate)

# start the tunnel here
metrics_port = randint(8100, 9000)
tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:7860',
     '--metrics', f'127.0.0.1:{metrics_port}'],
    stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
atexit.register(tunnel.terminate)

# finding the address here
tunnel_url = None
for _ in range(15):
    time.sleep(3)
    try:
        text = requests.get(f'http://127.0.0.1:{metrics_port}/metrics').text
        m = re.search(r'https?://[^\s"]+\.trycloudflare\.com', text)
        if m:
            tunnel_url = m.group(0)
            break
    except Exception:
        pass
# the website opens up here.
print(tunnel_url)


/content
Cloning into 'splat'...
remote: Enumerating objects: 149, done.
remote: Total 149 (delta 0), reused 0 (delta 0), pack-reused 149 (from 1)
Receiving objects: 100% (149/149), 79.38 KiB | 3.17 MiB/s, done.
Resolving deltas: 100% (82/82), done.
Selecting previously unselected package cloudflared.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.11.1) ...
Setting up cloudflared (2026.11.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
/usr/local/bin/cloudflared
https://restructuring-nomination-rendered-eur.trycloudflare.com
